# 01 · Access & Filter

**Stage 1 of 8** · ACCESS → WRANGLE → EXPLORE → VISUALIZE → ENGINEER → MODEL → EVALUATE → RECOMMEND

> **Project premise:** if someone has had cancer and was treated, **will it come back?** And for every factor linked to it coming back: **correlation or causation?**
>
> All data here is public: a k-anonymous feature table (every combination of site group, region, sex and diagnosis period is shared by ≥ 3 patients), synthetic patients, and aggregate reports. The raw database and clinical notes stay private.

**Goal of this stage:** load the data, understand its structure, and select the records that can answer the question.

**Business question:** Which patients and which columns do we need to measure how often cancer comes back?

In [1]:
%matplotlib inline
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
PUBLIC = ROOT / "data" / "public" / "patients_features_public.csv"
SYNTH = ROOT / "data" / "synthetic" / "synthetic_patients.csv"
BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8984"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e5e0", "axes.axisbelow": True, "figure.dpi": 110})
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
df = pd.read_csv(PUBLIC)

## Where the data comes from
The source is a MySQL dump of an oncology clinic's booking system (Sudan, 2020–2021). `src/parse_sql.py`
reads the dump straight into pandas, with no database server needed. The raw file is private,
so this notebook works on the **public, de-identified feature table** that the pipeline produces from it.

In [2]:
from parse_sql import load_dump   # how the raw dump is read (not run here: raw data is private)
quality = json.loads((ROOT / "reports" / "data_quality.json").read_text())
pd.DataFrame(quality["tables"], index=["rows", "columns"]).T.rename_axis("raw table")

,rows,columns
raw table,,
appointments,1270,5
patients,352,12
timeslots,42,3
unitdrs,6,8
users,1,2


### Q1 · How many rows and columns are in the dataset?

In [3]:
df.shape

(335, 54)

### Q2 · Which columns are available, and what data type does each contain?

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 335 entries, 0 to 334
Data columns (total 54 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   patient_id                 335 non-null    object 
 1   region                     335 non-null    object 
 2   ses_code                   332 non-null    object 
 3   site_group                 335 non-null    object 
 4   non_malignant_tumour       335 non-null    int64  
 5   sex                        335 non-null    object 
 6   elderly                    335 non-null    int64  
 7   years_since_dx             250 non-null    float64
 8   note_length                335 non-null    int64  
 9   note_missing               335 non-null    int64  
 10  histology                  335 non-null    object 
 11  grade                      93 non-null     float64
 12  er                         57 non-null     float64
 13  pr                         57 non-null     float64

Every row is **one patient** (335). Most columns are 0/1 flags extracted from the clinical notes
(treatments, metastasis sites), plus a few categories (`site_group`, `region`, `sex`, `breast_subtype`)
and the two outcome columns: `relapse` (did it come back?) and `rfs_years` (years until it did, or until the last note).

### Q3 · Display only the columns needed for the premise

In [5]:
cols = ["patient_id", "site_group", "sex", "region", "surgery", "radiotherapy", "hormonal_therapy",
        "metastatic_at_dx", "eligible_for_relapse", "relapse", "rfs_years"]
df[cols].head()

,patient_id,site_group,sex,region,surgery,radiotherapy,hormonal_therapy,metastatic_at_dx,eligible_for_relapse,relapse,rfs_years
0,P0324,Upper GI / hepatobiliary,Unknown,Central (Gezira / Sennar / White & Blue Nile),0,1,0,0,0,NaN,NaN
1,P0291,Genitourinary,Unknown,Suppressed,1,0,0,0,1,1.0,5.0
2,P0240,Skin,Unknown,Suppressed,0,1,0,0,1,0.0,6.0
3,P0223,Breast,F,Darfur,0,0,0,0,0,NaN,NaN
4,P0282,Genitourinary,M,Suppressed,0,1,1,1,0,NaN,NaN


### Q4 · Which patients have breast cancer? (Boolean filtering)

In [6]:
breast = df[df.site_group == "Breast"]
print(len(breast), "breast-cancer patients =", f"{len(breast)/len(df):.0%} of the clinic")

182 breast-cancer patients = 54% of the clinic


### Q5 · Which patients were already metastatic at diagnosis? (numeric condition)

In [7]:
denovo = df[df.metastatic_at_dx == 1]
print(len(denovo), "patients had spread at diagnosis. Their cancer never went away, so 'coming back' does not apply.")

64 patients had spread at diagnosis. Their cancer never went away, so 'coming back' does not apply.


### Q6 · Who can answer 'will it come back?' (AND logic)

In [8]:
at_risk = df[(df.non_malignant_tumour == 0) & (df.metastatic_at_dx == 0) &
             (df.eligible_for_relapse == 1)]
print(len(at_risk), "at-risk patients:", int(at_risk.relapse.sum()), "relapsed")

138 at-risk patients: 30 relapsed


The at-risk group uses **one rule for everyone**: malignant, not metastatic at diagnosis, treated,
and diagnosed at least a year before the note. (Version 1 of this project used a different rule for
patients with and without relapse. That bug is explained in notebook 05.)

### Q7 · Patients treated with radiotherapy OR hormonal therapy (OR logic) · `isin()` · `between()`

In [9]:
either = at_risk[(at_risk.radiotherapy == 1) | (at_risk.hormonal_therapy == 1)]
gyn_or_breast = at_risk[at_risk.site_group.isin(["Breast", "Gynaecologic"])]
followed_2_5 = at_risk[at_risk.rfs_years.between(2, 5)]
print(f"RT or HT: {len(either)} · breast/gynae: {len(gyn_or_breast)} · followed 2–5 yrs: {len(followed_2_5)}")

RT or HT: 109 · breast/gynae: 99 · followed 2–5 yrs: 49


### Selecting by position (`iloc`) and by label (`loc`)

In [10]:
display(at_risk.iloc[:3, :6])
at_risk.loc[at_risk.relapse == 1, ["site_group", "radiotherapy", "hormonal_therapy", "rfs_years"]].head()

,patient_id,region,ses_code,site_group,non_malignant_tumour,sex
1,P0291,Suppressed,V,Genitourinary,0,Unknown
2,P0240,Suppressed,V,Skin,0,Unknown
5,P0104,Khartoum,II,Breast,0,F


,site_group,radiotherapy,hormonal_therapy,rfs_years
1,Genitourinary,0,0,5.0
55,Breast,0,1,3.0
59,Breast,1,1,2.0
74,Breast,0,1,2.0
79,Breast,0,1,4.0


## So what?

To measure how often cancer comes back we need the **at-risk patients** (138 of 335), the outcome
(`relapse`, `rfs_years`) and the candidate factors (cancer type, treatments, biology, access). Patients who
were already metastatic at diagnosis (64) answer a different question (progression), not recurrence.